In [ ]:
import sys
from pathlib import Path

_root = next(p for p in Path.cwd().resolve().parents if (p / '.git').exists() or (p / 'setup.py').exists())
for _p in (str(_root), str(_root / 'src')):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import scanpy as sc

from metab_processing.metab_travlr_config import DATA_DIR, FOCUS_GENES
from metab_processing.SpaceTravLR.dataset_configs import dataset_paths
from metab_processing.SpaceTravLR.metab_loader import load_metabolites
from metab_processing.LinearRegression.build_x import build_x_adata, get_gene_factors

In [ ]:
data_dir = f'{DATA_DIR}/Alexi_UC_Spliced'
annot = '25_06_11_ICI_5K_Coarse_annotations'
focus_genes = list(FOCUS_GENES)
samples = [f'13473_HS4_UC-Slice_{i}' for i in (1, 2, 3, 4)]

In [ ]:
# Build one x_adata (all four factor groups + metabolites) per sample.
for dataset in samples:
    paths = dataset_paths(dataset, data_dir=data_dir)
    out_path = paths['dataset_dir'] / 'LinearRegression' / 'x_adata.h5ad'
    adata = sc.read_h5ad(paths['adata'])
    metabolites, _ = load_metabolites(paths['selection_yaml'], var_names=adata.var_names)
    print(f'=== {dataset}: {adata.n_obs} cells, {len(metabolites)} metabolite cols ===')
    proc = build_x_adata(adata, str(out_path), focus_genes=focus_genes,
                         metabolites=metabolites, annot=annot)
    print('  x_genes:', proc.uns['x_genes'])
    print('  x_factors:', proc.obsm['x_factors'].shape,
          '| x_metab:', proc.obsm.get('x_metab', None) if 'x_metab' in proc.obsm else None)

In [ ]:
# Quick check on the last-built sample: reconstruct one gene's matrix from the shared block.
g = proc.uns['x_genes'][0]
gx = get_gene_factors(proc, g, metabs='all')
print(f'get_gene_factors({g!r}, metabs="all") ->', gx.shape)
print(gx.columns[:10].tolist())